# Metadata

In [ ]:
# load metadata
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent.parent
RAW = ROOT / "data" / "symile"
OUT = RAW / "processed"

splits = {s: pd.read_csv(RAW / f"{s}.csv") for s in ("train", "val", "test")}
full = pd.read_csv(RAW / "symile_mimic_data.csv")
print({s: d.shape for s, d in splits.items()}, "| full:", full.shape)

In [ ]:
# test.csv holds 464 admissions repeated as 10 retrieval candidates each
pool = pd.concat([d[["hadm_id"]].assign(source_split=s) for s, d in splits.items()],
                 ignore_index=True).drop_duplicates("hadm_id").reset_index(drop=True)

assert len(pool) == 11214 and pool.hadm_id.is_unique
print(pool.source_split.value_counts().to_dict())

In [ ]:
# identity, target, admission context and the 50 raw labs, all from the full MIMIC table
LABS = [c for c in full.columns if c.isdigit()]
CONTEXT = ["age", "gender", "admission_type", "admission_location", "cxr_ViewPosition"]

joined = pool.merge(full, on="hadm_id", how="left", validate="one_to_one")
assert joined.subject_id.notna().all(), "admissions missing from the full table"

df = pd.DataFrame({
    "image_id": joined.hadm_id.astype(str),
    "patient_id": joined.subject_id.astype(str),
    "target": joined.hospital_expire_flag.astype("int8"),
    "source_split": joined.source_split,
    "cxr_dicom_id": joined.cxr_dicom_id,
    **{c: joined[c] for c in CONTEXT},
    **{f"lab_{c}": joined[c] for c in LABS},
})
print(df.shape, "| prevalence:", round(df.target.mean(), 4))


In [ ]:
# create train/test sets
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(splitter.split(df, y=df["target"], groups=df["patient_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

In [ ]:
overlap = set(train_df["patient_id"]) & set(test_df["patient_id"])
assert len(overlap) == 0, f"Patient leakage detected: {len(overlap)} patients overlap"

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))

print("\nTrain target distribution:")
print(train_df["target"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_df["target"].value_counts(normalize=True))

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(OUT / "train-metadata.parquet", index=False, engine="pyarrow")
test_df.to_parquet(OUT / "test-metadata.parquet", index=False, engine="pyarrow")

print("Saved:", OUT / "train-metadata.parquet")
print("Saved:", OUT / "test-metadata.parquet")

## Images

In [ ]:
# row r of cxr_<split>.npy is row r of <split>.csv; test repeats each admission, keep the first
import numpy as np

rows = {}
for s, d in splits.items():
    ids = np.load(RAW / "data_npy" / s / f"hadm_id_{s}.npy")
    assert np.array_equal(ids, d.hadm_id.to_numpy()), f"{s}: npy and csv rows disagree"
    first = {}
    for r, h in enumerate(ids):
        first.setdefault(str(int(h)), r)
    rows[s] = first

assert {i for r in rows.values() for i in r} == set(df.image_id)
print({s: len(r) for s, r in rows.items()})

In [ ]:
# the tensors are ImageNet-normalised: invert it and store the pixels losslessly
import io
from PIL import Image

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

def cxr_to_png_bytes(chw):
    grey = np.clip(np.asarray(chw) * STD + MEAN, 0.0, 1.0)[0]  # the three channels are identical
    buf = io.BytesIO()
    Image.fromarray(np.round(grey * 255).astype(np.uint8), "L").save(buf, "PNG", optimize=True)
    return buf.getvalue()

In [ ]:
# our split mixes rows from all three tensors, so read each one once with both stores open
import h5py
from tqdm import tqdm

SIZE = (320, 320)
dest = {**dict.fromkeys(train_df.image_id, "train"), **dict.fromkeys(test_df.image_id, "test")}

with h5py.File(RAW / "train-image.hdf5", "w") as h5_train, \
     h5py.File(RAW / "test-image.hdf5", "w") as h5_test:
    stores = {"train": h5_train, "test": h5_test}
    for h5 in stores.values():
        h5.attrs["format"] = "png_bytes"
        h5.attrs["image_size"] = str(SIZE)
        h5.attrs["id_col"] = "image_id"

    for s in splits:
        arr = np.load(RAW / "data_npy" / s / f"cxr_{s}.npy", mmap_mode="r")
        for image_id, r in tqdm(sorted(rows[s].items(), key=lambda kv: kv[1]), desc=f"cxr_{s}"):
            stores[dest[image_id]].create_dataset(
                image_id, data=np.bytes_(cxr_to_png_bytes(arr[r])))

In [ ]:
#sanity-check
row = train_df.sample(1).iloc[0]

with h5py.File(RAW / "train-image.hdf5", "r") as h5:
    assert set(h5.keys()) == set(train_df.image_id)
    png = h5[row.image_id][()]
with h5py.File(RAW / "test-image.hdf5", "r") as h5:
    assert set(h5.keys()) == set(test_df.image_id)

img = Image.open(io.BytesIO(png))
display(img)
print(img.size, img.mode, "| split:", row.source_split, "| target:", int(row.target))

src = np.load(RAW / "data_npy" / row.source_split / f"cxr_{row.source_split}.npy",
              mmap_mode="r")[rows[row.source_split][row.image_id]]
assert np.array_equal(np.array(img),
                      np.round(np.clip(np.asarray(src) * STD + MEAN, 0, 1)[0] * 255).astype(np.uint8)), \
    "the round trip is not lossless"
print("round trip is lossless")